# 🧠 Multimodal Identity RAG — Demo Notebook
### A personal document assistant that reasons across PDFs and images
**Stack:** LlamaIndex · LlamaParse · OpenRouter (GPT-4o-mini) · MultiModalVectorStoreIndex

---

### 0. Pre-flight Check — API Keys

NOTE: always ensure whatever documents your are providing in "../data" folder must not content name with spaces please avoid otherwise parser will ignore that file 

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('../.env')

# verify both keys are loaded — should print partial values not None
print("GOOGLE_API_KEY loaded:      ", "✅" if os.getenv("GOOGLE_API_KEY") else "❌ MISSING")
print("LLAMA_CLOUD_API_KEY loaded: ", "✅" if os.getenv("LLAMA_CLOUD_API_KEY") else "❌ MISSING")

### 1. Setup RAG Settings
Configures the global embedding model (`text-embedding-3-small` via OpenRouter)  
and initializes the Multimodal LLM (`gpt-4o-mini`) for querying.

In [ ]:
import sys
sys.path.append(os.path.abspath('../'))

from src.config import setup_rag_settings

# sets Settings.embed_model globally + returns mm_llm
mm_llm = setup_rag_settings()

print("RAG settings configured ✅")
print(f"Embedding model : BAAI/bge-small-en-v1.5 (local, HuggingFace)")
print(f"LLM             : gemini-2.5-flash (Gemini)")

### 2. The Parser Journey — Why Docling?

PDF parsing is the make-or-break step for RAG quality. Bad parsing means 
garbage embeddings, which means bad retrieval, no matter how good your LLM is.

I evaluated three parsers on the same set of certificates and degrees. 
The results below show why parser choice matters.

In [ ]:
# ── Attempt 1: PyMuPDF ──────────────────────────────────────────────
import fitz

doc = fitz.open("../data/Coursera_Structuring_ML_Projects.pdf")
raw_text = ""
for page in doc:
    raw_text += page.get_text()

print("── PyMuPDF output ──")
print(raw_text[:300])

# PyMuPDF returns vertical text as individual characters:
# "D e e p a k   R a t h o r e"   ← unusable for embeddings

**PyMuPDF problem:** Returns vertical text as spaced-out characters 
("D E E P A K"). Coursera/AWS certificates use decorative layouts with 
vertical headers — PyMuPDF can't reconstruct readable text from these. 
Embedding "D E E P A K R A T H O R E" matches nothing.

In [ ]:
# ── LlamaParse solution ─────────────────────────────────────────────
import os
import nest_asyncio
nest_asyncio.apply()

from llama_parse import LlamaParse

parser = LlamaParse(
    api_key=os.getenv("LLAMA_CLOUD_API_KEY"),
    result_type="markdown"
)

parsed_docs = parser.load_data("../data/Coursera_Structuring_ML_Projects.pdf")

print("── LlamaParse output ──")
print(parsed_docs[0].text[:500])


# ── Attempt 2: LlamaParse ───────────────────────────────────────────
# LlamaParse uses cloud AI parsing — handles vertical text correctly.
# However: known async event loop conflict in Jupyter causes 1-2 PDFs 
# to randomly fail per batch with "Event loop is closed" error.
# 
# We won't run it here — see Known Issues in README for full details.
# Below is what LlamaParse output looks like when it works:

llamaparse_sample_output = """
# Structuring Machine Learning Projects

Issued to: Deepak Rathore  
Issued by: DeepLearning.AI on Coursera  
Date: October 25, 2020  
Authorized by: Adjunct Professor Andrew Ng
"""
print(llamaparse_sample_output)

**LlamaParse problem:** Excellent output quality, but the async batch 
parsing conflicts with Jupyter's running event loop. Despite trying 
`nest_asyncio`, `num_workers=1`, and per-file `asyncio.run()` workarounds, 
1-2 PDFs randomly fail per batch (different files each run). 
Documented [LlamaParse GitHub issue](https://github.com/run-llama/llama_parse/issues).

**── Final choice: Docling (IBM Research, 2024) ──────────────────────
Fully synchronous, runs locally, handles complex layouts including 
vertical certificate text. No async event loop conflicts in Jupyter.

Drop-in replacement for LlamaParse via LlamaIndex's DoclingReader integration.**

from llama_index.readers.docling import DoclingReader

docling_parser = DoclingReader()
docs = docling_parser.load_data("../data/Coursera_Structuring_ML_Projects.pdf")

print("── Docling output ──")
print(docs[0].text[:400])

**Why Docling won:**

| Criterion | PyMuPDF | LlamaParse | Docling |
|---|---|---|---|
| Vertical text handling | ❌ | ✅ | ✅ |
| Markdown output | ❌ | ✅ | ✅ |
| Works in Jupyter | ✅ | ❌ (async bug) | ✅ |
| Runs locally (free) | ✅ | ❌ (cloud API) | ✅ |
| Production-ready | ⚠️ | ✅ | ✅ |

Docling is from IBM Research, released late 2024, now used in enterprise 
document pipelines at IBM, Red Hat, and others. It's the right choice 
for production document RAG.

### 3. Build Index (Run first time Only)
Loads all documents from `/data/` → parses PDFs via LlamaParse → builds MultiModalVectorStoreIndex → saves to disk.  
**Skip this cell after first run — use Load Index instead.**

In [ ]:
# force reload engine.py — Python caches imports, this picks up latest changes
import importlib
import src.engine
importlib.reload(src.engine)
from src.engine import build_index

# pre-download CLIP model so build_index doesn't crash kernel mid-way
import clip
print("Loading CLIP model...")
clip_model, _ = clip.load("ViT-B/32")
del clip_model
print("CLIP ready ✅\n")

# first run: parses docs + builds vectordb + saves to disk
index = build_index(
    doc_src_path="../data/",
    index_dest_path="../saved_index"
)

### 4. Load Index (Every Run After First)
Loads the saved index from disk — no re-parsing, no API calls, instant.

In [ ]:
from src.engine import load_index

# all runs after first: loads from disk in seconds
index = load_index(index_dest_path="../saved_index")

In [ ]:
# bypass Docling completely — read raw text with PyMuPDF to verify PDF integrity
import fitz

for pdf_name in ["Coursera_aws.pdf", "Coursera_Structuring_ML_Projects.pdf", "GCP.pdf"]:
    doc = fitz.open(f"../data/{pdf_name}")
    text = ""
    for page in doc:
        text += page.get_text()
    
    # check if text is readable
    printable_ratio = sum(1 for c in text[:500] if c.isprintable()) / max(len(text[:500]), 1)
    print(f"\n{'='*60}")
    print(f"FILE: {pdf_name}")
    print(f"Text extracted: {len(text)} chars")
    print(f"Printable ratio: {printable_ratio:.1%}")
    print(f"Preview: {text[:200]!r}")

In [ ]:
from llama_index.readers.docling import DoclingReader
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat

opts = PdfPipelineOptions()
opts.do_ocr = False
opts.do_table_structure = True
conv = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)})
parser = DoclingReader(doc_converter=conv)

for pdf_name in ["Coursera_aws.pdf", "Coursera_Structuring_ML_Projects.pdf", "GCP.pdf"]:
    try:
        docs = parser.load_data(f"../data/{pdf_name}")
        text = docs[0].text if docs else ""
        printable_ratio = sum(1 for c in text[:500] if c.isprintable()) / max(len(text[:500]), 1)
        print(f"\n{'='*60}")
        print(f"FILE: {pdf_name}")
        print(f"Docling extracted: {len(text)} chars, {printable_ratio:.1%} printable")
        print(f"Preview: {text[:300]!r}")
    except Exception as e:
        print(f"\n❌ {pdf_name}: {e}")

### 5. Inspect Nodes — What's Inside the Index?
Shows every chunk (node) stored in the vector database.  
Text nodes from PDFs + Image nodes from PNGs — all in one index.

In [ ]:
# force reload so latest engine.py is picked up
import importlib
import src.engine
importlib.reload(src.engine)
from src.engine import see_chunks

see_chunks(index)

### 6. Query the Index
Ask natural language questions — the engine retrieves relevant chunks  
and generates an answer using the Multimodal LLM.

In [ ]:
from src.engine import ask_query

# query 1 — text document
response = ask_query(index, mm_llm, "How many certificates do I have?")
print("Q: How many certificates do I have?")
print(f"A: {response}\n")

# query 2 — across image and text
response = ask_query(index, mm_llm, "Look at the image and tell me what document it is.")
print("Q: Look at the image and tell me what document it is.")
print(f"A: {response}\n")

# query 3 — specific fact
response = ask_query(index, mm_llm, "What is the graduation year from my degree?")
print("Q: What is the graduation year from my degree?")
print(f"A: {response}")

### 7. Agentic Tool Calling — LLM Uses Our Functions as Tools
The agent autonomously decides when to call `calculate_cosine_similarity`  
and `visualize_vectors` — we don't tell it when, it figures out the sequence itself.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

# import our own functions as tools for the agent
from experiments.exp_02_cosine_similarity import calculate_cosine_similarity, visualize_vectors

agent = FunctionAgent(
    tools=[calculate_cosine_similarity, visualize_vectors],
    llm=OpenAI(
        model="gpt-4o-mini",
        api_base="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENAI_API_KEY"),
        max_tokens=600
    ),
    system_prompt="You are a Vector Analysis Expert. Use the tools to calculate and visualize vectors."
)

# agent decides on its own: call calculate first, then visualize
response = await agent.run(
    user_msg="What is the cosine similarity between [-1,-2,-3] and [4,5,6]? Then visualize them."
)
print(response)

### 8. Context Memory — Agent Remembers the Conversation
`Context` object stores the full chat history.  
The agent can reference previous messages without us repeating the input.

In [ ]:
from llama_index.core.workflow import Context

# create context object — holds full conversation history
ctx = Context(agent)

# turn 1 — give the vectors
await agent.run(
    user_msg="Calculate cosine similarity between [-1,-2,-3] and [4,5,6]",
    ctx=ctx
)

# turn 2 — agent remembers the vectors from turn 1
response = await agent.run(
    user_msg="Now visualize those same vectors.",
    ctx=ctx
)
print(response)

# turn 3 — prove memory works
response = await agent.run(
    user_msg="What vectors did I give you in the first message?",
    ctx=ctx
)
print(response)

In [ ]:
# ── ACTUAL LlamaParse output ────────────────────────────────────────
# # C O U R S E  (decorative header — rendered as graphics in PDF)
# # C E R T I F I C A T E
#
# DEEPAK RATHORE  has successfully completed
# Structuring Machine Learning Projects
# an online non-credit course authorized by DeepLearning.AI  ← CLEAN
# Adjunct Professor Andrew Ng  ← CLEAN
# Coursera has confirmed the identity of this individual...  ← CLEAN
#
# Key insight: body text is clean and embeddable.
# PyMuPDF returns ALL text as spaced characters — including the body.
# LlamaParse correctly extracts the readable content that matters for RAG.